# 01 · Data and training (Colab)

Thin notebook: all logic lives in `src/`. Before running:
1. **Runtime → Change runtime type → T4 GPU.**
2. Push this repo to GitHub and put its URL in `REPO_URL` below.
3. *(Optional)* W&B logging: click the 🔑 **Secrets** icon on the left, add `WANDB_API_KEY`, enable notebook access. Without it, training still runs and logs to `history.csv`.

Checkpoints and logs go to Google Drive. If Colab disconnects, re-run the cells from the top: finished models are skipped and an interrupted one resumes from `last.pt`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv  # should say Tesla T4

In [ ]:
# Google Drive keeps checkpoints safe when the Colab VM is recycled.
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/plant-disease/runs'

In [ ]:
# Code lives on the fast local disk /content (not on Drive).
REPO_URL = 'https://github.com/Nagatejaswini12/plant-disease.git'
!git clone -q {REPO_URL} /content/plant-disease 2>/dev/null || git -C /content/plant-disease pull -q
%cd /content/plant-disease

In [ ]:
# Install our pinned requirements EXCEPT the ones Colab already ships with a matching CUDA build:
# reinstalling torch on Colab is slow and can break the GPU setup.
!grep -vE '^(torch|torchvision|numpy|pillow)==' requirements.txt > /tmp/req-colab.txt
!pip install -q -r /tmp/req-colab.txt

In [ ]:
# Optional W&B: read the key from Colab Secrets. `!python` commands below inherit this environment variable.
import os
try:
    from google.colab import userdata
    os.environ['WANDB_API_KEY'] = userdata.get('WANDB_API_KEY')
    print('W&B logging ON')
except Exception:
    print('No WANDB_API_KEY secret - W&B logging OFF (history.csv is still written)')

In [ ]:
# Download PlantVillage (2.2 GB zip) to /content/plant-disease/data and extract the colour images (~3 min).
!python -m src.data --images

## Smoke test
1 epoch on 64 train / 32 val images for all four models (~1 min). Run this first: it catches errors before you spend GPU hours. Results go to `runs/smoke/` and are not used later.

In [ ]:
!python -m src.train --model all --smoke --out {RUNS}

## Full training
One cell per model, in order. Each prints one line per epoch, including `seconds` so you can see the real speed. If the free GPU quota runs out, continue in a later session: re-run the setup cells, then the model cells.

In [ ]:
!python -m src.train --model custom_cnn --out {RUNS}

In [ ]:
!python -m src.train --model efficientnetv2_s --out {RUNS}

In [ ]:
!python -m src.train --model convnext_tiny --out {RUNS}

In [ ]:
!python -m src.train --model vit_small --out {RUNS}

In [ ]:
# Quick look: best validation macro-F1 per model (full evaluation on the test set is Phase 3).
import pandas as pd
for name in ['custom_cnn', 'efficientnetv2_s', 'convnext_tiny', 'vit_small']:
    path = f'{RUNS}/{name}/history.csv'
    if os.path.exists(path):
        h = pd.read_csv(path)
        best = h.loc[h.val_f1.idxmax()]
        print(f'{name:18s} best epoch {int(best.epoch):2d}  val acc {best.val_acc:.4f}  val macro-F1 {best.val_f1:.4f}  ({h.seconds.mean():.0f} s/epoch)')

## Evaluation (Phase 3)
Scores every `best.pt` on the held-out **test** split and writes `comparison.csv`, `comparison.png` and one confusion matrix per model to Drive. Copy that `reports/` folder into the repo afterwards.

In [ ]:
REPORTS = '/content/drive/MyDrive/plant-disease/reports'
!python -m src.evaluate --runs {RUNS} --out {REPORTS}